<a id="unit-u.07f0a424f37a4ffc8f8c6f28cbaed16c"></a>

<a id="table-models-for-observability-data"></a>

# Жүйені бақылау деректерінің кесте модельдері

**4-модуль: Жүйені бақылау деректерінің кесте модельдері** · [Сабақ](course.md) · [Тест](quiz04_observability_table_models.ipynb)

1-модульден бері дүкен тапсырыстарының шамамен жартысы сәтсіз болады. Инцидент кезінде де, кейін де бір сұрақтар қайталанады: әр сервис әр минутта қанша сұрау өңдеді, қаншасы сәтсіз болды, қандай қателер кездесті. Бұл зертханалық жұмыста жауаптарды ықшам кестелерде сақтап, кесте моделі Doris кілті бірдей жолдарға не істейтінін қалай шешетінін көресіз:

| Модель | Кілті бірдей жолдар | Жүйені бақылауда қолданылуы |
| --- | --- | --- |
| `DUPLICATE KEY` | Барлығы сақталады. Кілт бағандары тек сұрыптау ретін анықтайды. | Бастапқы журналдар, span-дар, метрика нүктелері |
| `AGGREGATE KEY` | Бір жолға бірігеді. Әр мән бағаны `SUM` сияқты агрегаттау функциясын атайды. | Алдын ала сақталған, мысалы сервис пен минут бойынша сұраулар мен ақаулар санақтары |
| `UNIQUE KEY` | Тек соңғы жүктелген жол сақталады. | Әр қате түрі мен күнге бір ескерту сияқты ағымдағы күй |

Экспорттаушы телеметрия кестелеріне қандай модельдер таңдағанын тексеріп, әр сервис пен минуттың сұраулары және сәтсіз сұраулары үшін AGGREGATE кестесін, әр қате түрі мен күнге бір ескерту сақтайтын UNIQUE кестесін құрасыз. Мысалдар `otel_traces` span-дарын оқиды. Жаңа кестелер `otel` дерекқорында құрылады; әр ұяшықты қайта орындауға болады.

Шамамен 35 минут жоспарлаңыз. Төлем ақауы қосылған 1-модульдің DOG стегі мен демосы қажет (1-зертханалық жұмыстың 5-бөлімі).

<a id="unit-u.60feadb0c6d14f8fa04d71e80ae4cbdd"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты инициализациялау

`lab.shell(...)` не `lab.sql(...)` қолданбас бұрын осы ұяшықты бір рет орындаңыз.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)

<a id="unit-u.5b05216f2a9f4d33845c255ea1542f94"></a>

<a id="1-check-the-failed-orders"></a>

## 1. Сәтсіз тапсырыстарды тексеру

Мысалдар `otel_traces` span-дарын, оның ішінде сәтсіз тапсырыс span-дарын қорытындылайды. Демо басталғалы тапсырыстарды және қаншасы сәтсіз болғанын санаңыз:

In [ ]:
orders = lab.wait_for_rows(
    """
    SELECT COUNT(*) AS orders,
           SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_orders,
           MIN(timestamp) AS first_order,
           MAX(timestamp) AS latest_order
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
    HAVING SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) > 0
    """,
    title="Orders and failed orders in otel_traces",
    hint="No order has failed. Check that the demo runs (lab.start_demo() in Module 1) and that paymentFailure "
         "is 50% (section 5 of Lab 1), wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)

<a id="unit-u.d204e5fec76f4368ab2d12e377c1ed63"></a>

**Күтілетін нәтиже**

Бір жол: демо басталғалы тапсырыстар, әдетте бірнеше ондықтан бірнеше жүзге дейін, олардың ішіндегі сәтсіздер — жалаушаны қосқаннан кейінгілердің шамамен жартысы, және алғашқы/соңғы тапсырыстың UTC уақыттары. Жалаушадан бері аз тапсырыс берілсе, кездейсоқтық сәтсіз үлесті жартыдан алыс етуі мүмкін.

<a id="unit-u.e4e6df45cef34fbd9b096f1b0f290edb"></a>

<a id="2-duplicate-keep-every-event"></a>

## 2. DUPLICATE: әр оқиғаны сақтау

Doris экспорттаушысы телеметрия кестелерін 1-модульде іске қосылғанда құрған. `information_schema.columns` әр кілт бағанын кесте моделімен белгілейді: DUPLICATE — `DUP`, AGGREGATE — `AGG`, UNIQUE — `UNI`. Осы кестелердің модельдері мен кілт бағандарын тізіңіз:

In [ ]:
lab.sql("""
SELECT TABLE_NAME AS table_name,
       MAX(COLUMN_KEY) AS model,
       GROUP_CONCAT(COLUMN_NAME ORDER BY ORDINAL_POSITION SEPARATOR ', ') AS key_columns
FROM information_schema.columns
WHERE TABLE_SCHEMA = 'otel' AND TABLE_NAME LIKE 'otel%' AND COLUMN_KEY <> ''
GROUP BY TABLE_NAME
ORDER BY table_name
""", title="Models and key columns of the telemetry tables");

<a id="unit-u.6ed1e1f2b49b4a0fb16a58b3a22f62d0"></a>

**Күтілетін нәтиже**

Сегіз кесте. Жетеуі — `DUP`: `otel_logs`, `otel_traces` және бес `otel_metrics_*` кестесі; кілті — `timestamp` пен `service_name` (кез келген ретпен). `otel_traces_graph` — `UNI`; себебін 4-бөлім түсіндіреді.

Журнал жолы, span және метрика нүктесі — жеке оқиғалар, сондықтан бастапқы кестелер DUPLICATE қолданады: Doris жүктеген әр жолды сақтайды. Кілт бағандары тек сұрыптау ретін анықтап, уақыт пен сервис бойынша сүзетін сұрауларға көмектеседі. `CREATE TABLE` кілт сөйлемі жоқ болса да DUPLICATE қолданады.

<a id="unit-u.f9a43cdfe9404d9baaa1e79a98570906"></a>

`otel_traces` кілті — `service_name` және span басталған уақыт `timestamp`. Бірнеше span-ға ортақ кілттерді іздеңіз:

In [ ]:
lab.sql("""
SELECT service_name, timestamp,
       COUNT(*) AS spans_with_this_key,
       COUNT(DISTINCT span_id) AS different_spans,
       COUNT(DISTINCT span_name) AS different_operations
FROM otel_traces
GROUP BY service_name, timestamp
HAVING COUNT(*) > 1
ORDER BY spans_with_this_key DESC, timestamp
LIMIT 5
""", title="Keys that several spans in otel_traces share");

<a id="unit-u.2ba366d47e0043c485c7e0661b691711"></a>

**Күтілетін нәтиже**

Әрқайсысы шамамен он span-ға ортақ бес кілт, әдетте дүкеннің серверлік және браузерлік коды `frontend` не `frontend-web`. SDK-лары уақытты миллисекундқа дейін тіркейді, бір сұрау бірнеше span-ды бір миллисекундта бастайды. `different_spans` мәні `spans_with_this_key` мәніне тең: әр span-ның өз ID-і бар, демек жолдар — бөлек оқиғалар; әр кілтке бір жол сақтайтын кесте біреуінен басқасын жоғалтар еді. DUPLICATE дәл көшірмелерді де сақтайды: бір деректі екі рет жүктеуден қорғамайды.

<a id="unit-u.1a4a33c47a7144498a648d2e826e6c92"></a>

<a id="3-aggregate-count-requests-and-failures-per-service-and-minute"></a>

## 3. AGGREGATE: сервис пен минут бойынша сұраулар мен ақауларды санау

Инцидент тақтасы әр сервис пен минутқа қанша сұрау өңделіп, қаншасы сәтсіз болғанын көрсетеді. Оны әр жаңаруда `otel_traces` кестесінен есептеу кезеңнің әр span-ын оқиды. AGGREGATE оның орнына санақтарды сервис пен минутқа бір жолмен сақтайды.

Сервис өңдеген сұрау — `span_kind` мәні `SPAN_KIND_SERVER` болатын сервер span-ы. `checkout` үшін бұлар тапсырыстар, `payment` үшін төлемдер. Келесі кестенің кілт бағандары есептің түйіршіктілігі: `request_minute` және `service_name`. `requests` пен `failed_requests` мән бағандары `SUM` жариялайды: кілті бұрыннан бар жол келгенде Doris мәндерін бұрынғы жолға қосады. Ол мұндай жолдарды жүктеу кезінде біріктіреді, қалғанын фонда біріктіреді (compaction), сұрау кезінде тағы қалғанын біріктіреді; сондықтан сұрау әр кілтке бір жол көреді.

Кілт span уақытын емес, минутты сақтайды. Span уақыты кілтте болса, тек бір микросекунд span-дары бірігіп, кесте `otel_traces` сияқты үлкен болар еді. Әр сұрау аяқталған минутта, `DATE_TRUNC(end_time, 'minute')`, саналады: span аяқталғаннан кейін ғана Doris-ке жетеді; `flagd` жалауша ағындары сияқты 10 минутқа дейін ашық ұзақ сұрау басталған минут саналғаннан көп кейін келер еді. `DISTRIBUTED BY HASH(service_name)` бір сервистің барлық жолын бір tablet-ке жібереді; осындай кішкентай кестеге бір bucket жеткілікті. 5-модуль бөлімдер мен bucket-терді таңдауды түсіндіреді.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS requests_per_minute (
    request_minute  DATETIME,
    service_name    VARCHAR(200),
    requests        BIGINT SUM,
    failed_requests BIGINT SUM
)
AGGREGATE KEY(request_minute, service_name)
DISTRIBUTED BY HASH(service_name) BUCKETS 1
""", title="Create the AGGREGATE table requests_per_minute");

<a id="unit-u.31e806ca126d4d72983fbca5d640fa50"></a>

**Күтілетін нәтиже**

Карточка команданың сәтті орындалғанын айтады. `IF NOT EXISTS` бұрынғы кестені өзгертпейді, сондықтан ұяшықты қайталауға болады.

<a id="unit-u.89f8ed2715ba4fa0aa5a6b184409a1b8"></a>

<a id="load-the-spans-in-two-batches"></a>

### Span-дарды екі пакетпен жүктеу

Жүктеу тапсырмасы әдетте жоспар бойынша орындалып, жаңа аяқталған кезең жолдарын қосады. Келесі ұяшық `otel_traces` сервер span-дарын `requests_per_minute` кестесіне осындай екі пакетпен жүктейді: екіншісі 30 секундтық кезеңде аяқталған span-дар, біріншісі — одан бұрын аяқталғанның бәрі. Span-дар аяқталғаннан бірнеше секунд кейін Doris-ке жететіндіктен, ұяшық ең жаңа span алдындағы соңғы 30 секундты қоспайды.

Әр span `requests = 1` және күйі `STATUS_CODE_ERROR` болса `failed_requests = 1`, әйтпесе `0` мәнді бір жолға айналады. Әр пакеттен кейін ұяшық екінші пакет басталған минуттың кесте жолдарын көрсетеді. Орындамас бұрын болжаңыз: екінші пакет сол минутқа жол қоса ма, әлде бар жолдарды өзгерте ме?

In [ ]:
bounds = lab.sql("""
SELECT MAX(end_time) - INTERVAL 60 SECOND AS second_batch_start,
       MAX(end_time) - INTERVAL 30 SECOND AS second_batch_end,
       DATE_TRUNC(MAX(end_time) - INTERVAL 60 SECOND, 'minute') AS boundary_minute
FROM otel_traces
""", title="Time range of the second batch")
second_batch_start = bounds["second_batch_start"][0]
second_batch_end = bounds["second_batch_end"][0]
boundary_minute = bounds["boundary_minute"][0]
lab.execute("TRUNCATE TABLE requests_per_minute", title="Empty the table, so that the cell can be rerun")

LOAD_REQUESTS = """
INSERT INTO requests_per_minute (request_minute, service_name, requests, failed_requests)
SELECT DATE_TRUNC(end_time, 'minute'), service_name, 1, IF(status_code = 'STATUS_CODE_ERROR', 1, 0)
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER' AND {period}
"""
BOUNDARY_ROWS = f"""
SELECT * FROM requests_per_minute
WHERE request_minute >= '{boundary_minute}'
ORDER BY request_minute, requests DESC
"""

lab.execute(LOAD_REQUESTS.format(period=f"end_time < '{second_batch_start}'"),
            title="First batch: the server spans that ended before the second batch")
lab.sql(BOUNDARY_ROWS, title="requests_per_minute from the minute in which the second batch starts, after the first batch")

lab.execute(LOAD_REQUESTS.format(period=f"end_time >= '{second_batch_start}' AND end_time < '{second_batch_end}'"),
            title="Second batch: the server spans that ended in 30 seconds")
lab.sql(BOUNDARY_ROWS, title="The same rows after the second batch");

<a id="unit-u.ae9874fc02284a328d6b851c1e27c3b1"></a>

**Күтілетін нәтиже**

Алдымен екінші пакет кезеңі: ең жаңа span-нан 30 секунд бұрын аяқталатын 30 секунд және ол басталған минут. Бірінші пакет жүктеген әр span-ына бір әсер еткен жол, яғни ондаған мың жол хабарлайды; бірақ `requests_per_minute` ішінде сервис пен минутқа бір жол ғана бар. Біріншіден кейін шекаралық минут жолдары оның `second_batch_start` алдындағы алғашқы секундтарының сұрауларын сақтайды.

Екінші пакет бірнеше жүзден бірнеше мыңға дейін span жүктейді. Кейін шекаралық минутта әлі әр сервиске бір жол бар: `frontend-proxy` не `frontend` сияқты екі пакетте де сұрау өңдеген сервис өз жолын сақтап, `requests` мәні екінші пакет span-дарының санына өседі. Бұл — `SUM` біріктіруі: бір кілтке 4, кейін 7 жүктеу 11 мәнді бір жол береді. Минуттағы алғашқы сұрауы екінші пакетте келген сервис жаңа жол алады; пакет келесі минутқа өтсе, сол минуттың әр сервисі де жаңа жол алады.

AGGREGATE жаңа жолды қайталанған жүктеуден ажырата алмайды. Екінші пакет екі рет жүктелсе, span-дары екі рет қосылып, санақтар артық болар еді. Тапсырма әр бастапқы жолды дәл бір рет жүктеуі керек; бұл ұяшық `TRUNCATE TABLE` арқылы басталады, сондықтан оны қайталауға болады.

<a id="unit-u.03935c0afdc84cf58e6925ac67d963e0"></a>

Санақтарды бастапқы span-дармен салыстырыңыз. Келесі сұрау екінші пакеттің соңына дейін `otel_traces` сервер span-дарын сервис пен минут бойынша санап, әр санақты `requests_per_minute` жолымен салыстырады:

In [ ]:
lab.sql(f"""
SELECT COUNT(*) AS groups_in_otel_traces,
       SUM(IF(s.requests = r.requests AND s.failed_requests = r.failed_requests, 1, 0)) AS groups_that_match,
       (SELECT COUNT(*) FROM requests_per_minute) AS rows_in_requests_per_minute,
       SUM(r.requests) AS server_spans_read
FROM (
    SELECT DATE_TRUNC(end_time, 'minute') AS request_minute, service_name,
           COUNT(*) AS requests,
           SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_requests
    FROM otel_traces
    WHERE span_kind = 'SPAN_KIND_SERVER' AND end_time < '{second_batch_end}'
    GROUP BY DATE_TRUNC(end_time, 'minute'), service_name
) AS r
LEFT JOIN requests_per_minute AS s
  ON s.request_minute = r.request_minute AND s.service_name = r.service_name
""", title="requests_per_minute checked against the server spans in otel_traces");

<a id="unit-u.f6c91a18bfd1480b9c8765e095090ea4"></a>

**Күтілетін нәтиже**

Алғашқы үш бағанында бірдей сан бар бір жол: әр сервис пен минутқа бір топ, әр топ `requests_per_minute` жолына сәйкес, кестеде артық жол жоқ. `server_spans_read` — санақтарды есептеуге оқылған ондаған мың span саны; `requests_per_minute` сол жауапты сервис пен минутқа бір жолдан береді. Құны — кесте тек өз түйіршіктілігіндегі сұрақтарға жауап береді: қай сұраулар сәтсіз болғанын не қанша уақыт алғанын айта алмайды.

`groups_that_match` кіші болса, кезеңнің кей span-дары Doris-ке жоғарыдағы ұяшық жүктегеннен кейін келген: екі ұяшықты да қайталаңыз.

<a id="unit-u.09eb8d70d6574a6abb3ffd0421b6c70f"></a>

Енді инцидентті қорытындыдан оқыңыз: әр минуттың тапсырыстары `checkout` және төлемдері `payment`, сәтсіздерімен бірге.

In [ ]:
lab.sql("""
SELECT request_minute, service_name, requests, failed_requests,
       ROUND(100 * failed_requests / requests) AS failed_percent
FROM requests_per_minute
WHERE service_name IN ('checkout', 'payment')
ORDER BY request_minute DESC, service_name
LIMIT 20
""", title="Orders and charges per minute, newest first");

<a id="unit-u.38770f2fbe2c4682af7f2828122cab87"></a>

**Күтілетін нәтиже**

20 жолға дейін, көп минутқа екеуден: `checkout` минутына бір-төрт тапсырысты, `payment` төлемдерді санайды. Әр тапсырыс бір төлем жасайды, сондықтан сандар әдетте тең; тапсырыс төлемінен сәл кейін келесі минутта аяқталса, айырмасы бір болады. Жалаушаны қосқалы төлемдердің шамамен жартысы, олармен бірге тапсырыстары сәтсіз: екі сервистен көрінген бір ақаулар. Тапсырыс аз болғандықтан, бір минутта ешқайсысы не барлығы сәтсіз болуы мүмкін.

<a id="unit-u.756ad8afbedd452da779deffbdba9c6b"></a>

<a id="4-unique-one-alert-per-kind-of-error-and-day"></a>

## 4. UNIQUE: әр қате түрі мен күнге бір ескерту

Тапсырма әр минутта қате span-дарын оқып, әрқайсысына ескерту жасайды делік. Әр сәтсіз тапсырыс бірнеше сервисте қате span-ын қалдырады, сондықтан әр span-ға бір ескерту — бір мәселе туралы минутына бірнеше ескертуден тұратын тасқын. UNIQUE әр кілтке бір жол сақтайды; кілт күн мен ескерту сәйкестендіргіші болса, жиілігіне қарамастан әр қате түрі күніне бір ескерту береді.

Сәйкестендіргіш бір қате түрінің әр қайталануында бірдей, түрлер арасында әртүрлі болуы керек. Мұнда ол — сервис, операция (`span_name`) және span күй хабары. Хабарларда адрес, порт, ID сияқты әр оқиғада өзгеретін бөліктер болуы мүмкін. Сондықтан төмендегі сұрау хабардың әр санын `0` етіп, нәтижесін сервис пен операциямен бірге хештейді: `alert_hash`. Қате span-дарын қалай топтайтынын қараңыз:

In [ ]:
alerts = lab.wait_for_rows(
    r"""
    SELECT DATE(timestamp) AS alert_date,
           service_name,
           span_name,
           MD5(CONCAT_WS('|', service_name, span_name, REGEXP_REPLACE(status_message, '[0-9]+', '0'))) AS alert_hash,
           COUNT(*) AS error_spans,
           MAX(end_time) AS last_error,
           LEFT(MIN(REGEXP_REPLACE(status_message, '[0-9]+', '0')), 80) AS normalized_message
    FROM otel_traces
    WHERE status_code = 'STATUS_CODE_ERROR'
    GROUP BY alert_date, service_name, span_name, alert_hash
    ORDER BY error_spans DESC, alert_hash
    """,
    title="Error spans by alert identity",
    hint="otel_traces has no error spans. Turn on the payment failure with section 5 of Lab 1, wait for "
         "the first failed order, then rerun this cell.",
    timeout_seconds=10,
)

<a id="unit-u.646e075d0fd546bcb8f79df106b40b20"></a>

**Күтілетін нәтиже**

Ескерту мен күнге бір жол, әдетте 15–20. Инциденттің қате түрлері алдымен, әрқайсысында бір сәтсіз тапсырысқа бір қате span-ы бар: payment `grpc.oteldemo.PaymentService/Charge`, checkout `oteldemo.CheckoutService/PlaceOrder` және оның payment шақыруы `oteldemo.PaymentService/Charge`, сондай-ақ сәтсіз сұрауды тасымалдаған `frontend`, `frontend-proxy`, `load-generator` span-дары. Одан төмен демо басталған минуттағы бір қате span-ы бар түрлер: бұлттық метадерек сервисі `metadata.google.internal` адресіне жете алмаған `payment` пен `frontend`, әлі жоқ Kafka тақырыбын оқыған `accounting`. `normalized_message` сандар неге алмастырылатынын көрсетеді: `recommendation` жалауша жаңартуларының `/flagd.evaluation.v1.Service/EventStream` ағыны әр 10 минут сайын уақыты бар хабармен `DEADLINE_EXCEEDED` болып аяқталады; сандар алмастырылса, әр қайталану бір сәйкестендіргіш алады. Демо 10 минут жұмыс істегенде бұл жол пайда болады. `last_error` — ескертудің соңғы span-ы аяқталған уақыт.

<a id="unit-u.b946e03da7154789933fbe43fd515545"></a>

Ескерту кестесін құрыңыз. Кілті — күн мен ескерту сәйкестендіргіші:

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS error_alerts (
    alert_date   DATE,
    alert_hash   VARCHAR(32),
    service_name VARCHAR(200),
    span_name    STRING,
    status       VARCHAR(16),
    message      STRING
)
UNIQUE KEY(alert_date, alert_hash)
DISTRIBUTED BY HASH(alert_hash) BUCKETS 1
""", title="Create the UNIQUE table error_alerts");

<a id="unit-u.863676d2f5604641b8ebab93627f2232"></a>

**Күтілетін нәтиже**

Карточка команданың сәттілігін айтады. Кілті бірдей жолдардың бәрі бір tablet-ке жетуі үшін UNIQUE кілт бағандарының хешімен бөлінуі керек. Doris 4.x UNIQUE кестелерін әдепкі Merge-on-Write (MoW) ретінде құрады: кілті бұрыннан бар жол келгенде жаңа жолды жүктеп жатып ескісін жойылған деп белгілейді, сондықтан сұраулар тек ең жаңа жолды оқиды.

<a id="unit-u.b5f88c71f6e14c5ca5a85d2a20a73c19"></a>

Тапсырма сияқты әр қате span-ына ашық ескерту жасаңыз:

In [ ]:
RAISE_ALERTS = r"""
INSERT INTO error_alerts (alert_date, alert_hash, service_name, span_name, status, message)
SELECT DATE(timestamp),
       MD5(CONCAT_WS('|', service_name, span_name, REGEXP_REPLACE(status_message, '[0-9]+', '0'))),
       service_name,
       span_name,
       'open',
       status_message
FROM otel_traces
WHERE status_code = 'STATUS_CODE_ERROR'
"""
lab.execute("TRUNCATE TABLE error_alerts", title="Empty the table, so that the cell can be rerun")
lab.execute(RAISE_ALERTS, title="Raise an open alert for every error span")
lab.sql("SELECT * FROM error_alerts ORDER BY service_name, span_name", title="error_alerts");

<a id="unit-u.042805cbba0a43ed9cc07aa8d1493e9e"></a>

**Күтілетін нәтиже**

Кірістіру әр қате span-ына бір әсер еткен жол, әр сәтсіз тапсырысқа тоғыз және бірнеше басқа жол хабарлайды; бірақ `error_alerts` ішінде алдыңғы сұрау қайтарғандай ескерту мен күнге бір жол бар: кілті бірдей жолдар бірін-бірі алмастырды. Әр жол бір оқиғаның, Doris соңғы жүктегенінің, хабарын сақтайды.

Бір жол бір қатені білдірмейді. Payment төлемінің ескертуі `error_alerts` ішінде бір жол, `otel_traces` ішінде әр сәтсіз тапсырысқа бір қате span-ын береді. Кесте санақты емес, ағымдағы күйді сақтайды: жиілікті білу үшін алдыңғы сұраудай `otel_traces` span-дарын санаңыз не 3-бөлімдегідей AGGREGATE ішінде санақ сақтаңыз.

<a id="unit-u.8de21022308b4edb93a02bfbd89f6f6c"></a>

Бір минуттан кейін тапсырма қайта орындалып, сол қате span-дарын оқиды. Оларды тағы жүктеңіз:

In [ ]:
lab.execute(RAISE_ALERTS, title="Raise the alerts again from the same error spans")
lab.sql("SELECT COUNT(*) AS alerts FROM error_alerts", title="Alerts after the second load");

<a id="unit-u.9890dc064a314ff1b273fc620bd3d924"></a>

**Күтілетін нәтиже**

Кірістіру сол жол санын және арада сәтсіз болған тапсырыстардың қате span-дарын қайта жүктейді; `alerts` өзгерген жоқ. Қайталанған жүктеу жолдарды қоспай, алмастырады, сондықтан UNIQUE кестесін қауіпсіз қайта жүктеуге болады. 3-бөлімдегі `requests_per_minute` қайта жүктелсе, әр span екі рет саналар еді.

<a id="unit-u.379d75a417554fd28cbf5dd3f77bd32f"></a>

Мәселе шешілгенде тапсырма ескертуін `resolved` күйімен қайта жазады. Төлем ақауы жалғасады, бірақ демоның іске қосылу қателері бір рет қана болды. Соңғы қатесі ең ескі ескертуді шешіңіз:

In [ ]:
oldest = alerts.sort_values("last_error").iloc[0]
print(f"Resolve the alert of {oldest['service_name']} {oldest['span_name']}, "
      f"last error at {oldest['last_error']} UTC")
lab.execute(f"""
INSERT INTO error_alerts (alert_date, alert_hash, service_name, span_name, status, message)
SELECT alert_date, alert_hash, service_name, span_name, 'resolved', message
FROM error_alerts
WHERE alert_date = '{oldest['alert_date']}' AND alert_hash = '{oldest['alert_hash']}'
""", title="Write the alert again with the status resolved")
lab.sql(f"SELECT * FROM error_alerts WHERE alert_hash = '{oldest['alert_hash']}'", title="The resolved alert")
lab.sql("SELECT status, COUNT(*) AS alerts FROM error_alerts GROUP BY status ORDER BY status", title="Alerts by status");

<a id="unit-u.753534cfc5b942ac96b686cadb7a14a6"></a>

**Күтілетін нәтиже**

Алдымен ұяшық шешетін ескерту, әдетте демо іске қосылған кезден. Кірістіру бір жол жазады; ескертуде әлі бір жол бар, енді `status = resolved`. `Alerts by status` бұрынғыдай ескерту санын береді, біреуі шешілген: сол кілттің кейінгі жүктеуі ашық жолды алмастырды.

Ескерту жасау ұяшығын енді қайта орындасаңыз, бұл ескерту қайтадан `open`, өйткені ол жүктеу одан да кейін: UNIQUE соңғы жүктелген жолды сақтайды. Жүктеулер ретсіз келуі мүмкін болса, кестеге sequence бағанын (`function_column.sequence_col` қасиеті) беріңіз: Doris жүктеу ретіне қарамастан сол бағандағы ең үлкен мәнді, мысалы ескертудің соңғы қатесінің уақытын сақтайды. Шешілген уақытпен жазылған жол сол уақыттан жаңа қате жүктелгенше қалады.

<a id="unit-u.b7d6f5c1defd435aa5220c527270b95c"></a>

<a id="the-exporters-unique-table"></a>

### Экспорттаушының UNIQUE кестесі

2-бөлімдегі UNIQUE кесте `otel_traces_graph` сервистер арасындағы минуттық шақыруларды санайды. Экспорттаушы құрған `otel_traces_graph_job` оны әр 10 минут сайын толтырады: әр орындалу `otel_traces` span-дарын 10 минут бұрынғы минуттың басынан қазіргі уақытқа дейін санайды. Сондықтан әр орындалу алдыңғы орындалу басталған, сол кезде тек ішінара көрінген минутты қайта санап, кілті бірдей жолдарды жазады. UNIQUE бұрынғы ішінара санақтарды жаңаларымен алмастырады. AGGREGATE `SUM` болса екі санақ қосылар еді, DUPLICATE болса минут екі рет көрінер еді.

Жоспарлы кірістіру тапсырмаларын тізетін `jobs('type'='insert')` арқылы тапсырманы және соңғы минуттардағы `frontend-proxy` → `frontend` шақыруларын қараңыз:

In [ ]:
graph_job = lab.sql("""
SELECT Name, RecurringStrategy, Status, SucceedTaskCount, LastTaskSuccessTime
FROM jobs('type'='insert')
WHERE Name LIKE '%otel_traces_graph_job'
""", title="The job that fills otel_traces_graph")
lab.sql("""
SELECT timestamp, caller_service_name, callee_service_name, count, error_count
FROM otel_traces_graph
WHERE caller_service_name = 'frontend-proxy' AND callee_service_name = 'frontend'
ORDER BY timestamp DESC
LIMIT 5
""", title="Calls from frontend-proxy to frontend per minute");

<a id="unit-u.1a3e12cc4cb44d1a92b7255511d01488"></a>

**Күтілетін нәтиже**

Тапсырма `EVERY 10 MINUTE` бойынша орындалады, `SucceedTaskCount` қанша рет орындалғанын көрсетеді. Көп минутта шақыру саны ұқсас, демо қосулы кезде шамамен 1,450. Ең жаңа минуттарда аз не әлі жол жоқ болуы мүмкін: соңғы орындалу span-дарының бәрі Doris-ке жетпей санап қойған. Келесі екі ұяшық бір минутты бекітеді, сонда ол ең жаңа бес жолдан шыққаннан кейін де табылады.

`error_count` сәтсіз шақыруларды санайды. Инцидент `checkout` → `payment` шақыруларында көрінеді: екінші сұраудағы екі сервис атауын өзгертіп қайталаңыз; жалаушаны қосқалы әр минуттағы шақырулардың шамамен жартысы сәтсіз.

`SucceedTaskCount` — 0, екінші кесте бос болса, тапсырма әлі орындалмаған. DOG Collector схеманы `create_schema: true` арқылы құрады: оны қайта қосу тапсырманы жойып қайта құрады, алғашқы орындалуы 10 минуттан кейін. 2-модуль трасса collector-ында `create_schema: false`; оны қайта қосу тапсырманы қайта құрмайды. Алғашқы орындалу бітпесе, біткен соң жоғарыдағы тапсырма тексеруі мен төмендегі бастапқы мән алу ұяшығын қайталаңыз.

<a id="unit-u.d76c39f7d9284668987b0ed0df90765b"></a>

`LastTaskSuccessTime` минутын есте сақтау үшін осы ұяшықты **тапсырма сәтті орындалғаннан кейін бір рет** орындаңыз. Нысана мен ағымдағы санын шығарады. Демоны қосулы қалдырыңыз. Келесі жоспарлы орындалудан кейін есте сақталған минут өзгермес үшін төмендегі салыстыру ұяшығын ғана қайталаңыз. Ядроны қайта іске қосқаннан кейін алдымен жаңа бастапқы мән алыңыз.

In [ ]:
from datetime import datetime

if int(graph_job["SucceedTaskCount"].iloc[0]) > 0:
    graph_minute = datetime.fromisoformat(str(graph_job["LastTaskSuccessTime"].iloc[0])).replace(second=0, microsecond=0)
    GRAPH_MINUTE_QUERY = f"""
    SELECT timestamp, caller_service_name, callee_service_name, count, error_count
    FROM otel_traces_graph
    WHERE timestamp = '{graph_minute:%Y-%m-%d %H:%M:%S}'
      AND caller_service_name = 'frontend-proxy' AND callee_service_name = 'frontend'
    """
    graph_before = lab.sql(GRAPH_MINUTE_QUERY, title=f"Baseline for the fixed minute {graph_minute} UTC")
else:
    graph_minute = None
    print("The graph job has not completed a run. Wait for its first run, then rerun the job check and this capture cell.")

<a id="unit-u.f059870ec23f43e28cb0da204672a243"></a>

Келесі тапсырма орындалған соң осы салыстыру ұяшығын қайта орындаңыз. Тапсырма тексеруі соңғы орындалудың қашан аяқталғанын көрсетеді; екінші сұрау алынған минут соңғы бес жолдан шығып кетсе де, әрдайым сол минутты таңдайды. Сол сервис жұбын саны толтырылған бір жолда күтіңіз; сол кіріс span-дары сақталғанша сан кемімеуі керек.

In [ ]:
if graph_minute is None:
    print("Capture a baseline after the graph job's first successful run before comparing it.")
else:
    lab.sql("""
    SELECT Name, SucceedTaskCount, LastTaskSuccessTime
    FROM jobs('type'='insert')
    WHERE Name LIKE '%otel_traces_graph_job'
    """, title="Check whether the next scheduled graph run has finished")
    lab.sql(GRAPH_MINUTE_QUERY, title=f"Compare the same minute {graph_minute} UTC with the baseline above");

<a id="unit-u.6e4348c5f14647f5b4f5429d8888a0a2"></a>

<a id="5-exercise-a-daily-summary-of-the-logs"></a>

## 5. Жаттығу: журналдардың күндік қорытындысы

Журналдар үшін ұқсас қорытынды құрыңыз: кілті `log_date` пен `service_name`, `SUM` бағандары `log_count` пен `error_count` болатын `daily_log_counts`. Журнал жолының `severity_number` мәні 17 не одан жоғары, OpenTelemetry ERROR/FATAL аралығында болса, ол қате. Оны `otel_logs` кестесінен бекітілген уақытқа, ең жаңа журнал жолынан 30 секунд бұрынғы сәтке дейін жүктеп, сол шекті қолданатын `otel_logs` бойынша `GROUP BY` сұрауымен салыстырыңыз. Сервис пен күнге бір жол, екі нәтижеде бірдей сандар күтіңіз. Қай сервистерде қате жолдары бар, төлем ақауы неге ештеңе қоспайды?

In [ ]:
# Your code here

<a id="unit-u.bfc2b50d2f3940939f7f24daafec6ad6"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
log_limit = lab.sql(
    "SELECT MAX(timestamp) - INTERVAL 30 SECOND AS log_limit FROM otel_logs",
    title="Load the log rows before this time",
)["log_limit"][0]

lab.execute("""
CREATE TABLE IF NOT EXISTS daily_log_counts (
    log_date     DATE,
    service_name VARCHAR(200),
    log_count    BIGINT SUM,
    error_count  BIGINT SUM
)
AGGREGATE KEY(log_date, service_name)
DISTRIBUTED BY HASH(service_name) BUCKETS 1
""", title="Create the AGGREGATE table daily_log_counts")
lab.execute("TRUNCATE TABLE daily_log_counts", title="Empty the table, so that the cell can be rerun")
lab.execute(f"""
INSERT INTO daily_log_counts (log_date, service_name, log_count, error_count)
SELECT DATE(timestamp), service_name, 1, IF(severity_number >= 17, 1, 0)
FROM otel_logs
WHERE timestamp < '{log_limit}'
""", title="Load the log rows")
lab.sql("SELECT * FROM daily_log_counts ORDER BY log_count DESC", title="daily_log_counts")
lab.sql(f"""
SELECT DATE(timestamp) AS log_date, service_name,
       COUNT(*) AS log_count,
       SUM(IF(severity_number >= 17, 1, 0)) AS error_count
FROM otel_logs
WHERE timestamp < '{log_limit}'
GROUP BY DATE(timestamp), service_name
ORDER BY log_count DESC
""", title="The same counts computed from otel_logs");

<a id="unit-u.fd3d9bd07be24226814fc7204458f010"></a>

**Күтілетін нәтиже**

Екі кестеде жолдар мен сандар бірдей, шамамен 11 сервис: `frontend-proxy` және 2-модульдің FE журналы `doris-fe` жолдары ең көп. Қате жолдары аз сервисте бар: FE іске қосылғанда жазған қателерден `doris-fe`, кейде `accounting` сияқты демо сервисі. Төлем ақауы ештеңе қоспайды: payment пен checkout журнал жібермейді, ал 2-модуль көрсеткендей, сәтсіз checkout прокси қатынас журналында маңыздылық жоқ. Қате журналдарының қорытындысы бұл инцидентті өткізіп алар еді; span-дардан құрылған 3-бөлім қорытындысы оны көрсетеді.

<a id="unit-u.cc9d7fba65f4470692e7e94e8b98794c"></a>

<a id="you-are-done-when"></a>

## Аяқтау шарттары

- экспорттаушы бастапқы телеметрияға қай модель қолданатынын және `otel_traces` ішінде кілті ортақ span-дардың бәрі неге сақталуы керегін айта аласыз;
- `requests_per_minute` ішінде сервис пен минутқа бір жол бар, екінші пакет бар жолдарға қосылды, санақтар `otel_traces` сервер span-дарымен сәйкес;
- `error_alerts` ішінде қате не жүктеу жиілігіне қарамастан ескерту мен күнге бір жол бар, шешкен ескертуіңіз `resolved` көрсетеді;
- `otel_traces_graph` неге UNIQUE екенін түсіндіре аласыз.

Жұмыс кестелері кішкентай; кейінгі модульдер оларды қолданбайды. Жою үшін `lab.execute(...)` не MySQL клиентімен `DROP TABLE IF EXISTS requests_per_minute`, `DROP TABLE IF EXISTS error_alerts` және `DROP TABLE IF EXISTS daily_log_counts` орындаңыз.

<a id="unit-u.e3b113e69aaf4b64bce109035f53019d"></a>

<a id="next"></a>

## Әрі қарай

[4-тест: Жүйені бақылау деректерінің кесте модельдері](quiz04_observability_table_models.ipynb) файлын ашып, [5-модуль: Жүйені бақылау деректерін бөлімдеу және bucket-терге бөлу](../module05-partition-bucket/course.md) бөліміне өтіңіз.